# Lab 4 — Retrieval-Augmented Generation (RAG) Pipeline with Local Models

Language models have a frozen worldview: they only know what was in their pretraining corpus. Ask GPT-2 about a 2024 event and it hallucinates. **Retrieval-Augmented Generation** fixes this at the cheapest possible place in the stack — you look up relevant facts and paste them into the prompt. No fine-tuning, no retraining, just **embed + search + prompt**.

RAG is the single most widely-deployed LLM pattern in 2024–2026. Every enterprise "chat with your docs" product, every helpdesk bot, every search-grounded answer in ChatGPT and Claude and Gemini — all RAG underneath. The [Lewis et al. 2020 paper](https://arxiv.org/abs/2005.11401) introduced the term; the whole ecosystem (LangChain, LlamaIndex, pgvector, Milvus, Weaviate, Pinecone) exists to make this one pipeline easier to build.

### The pipeline in five moves

```
documents → [chunker] → chunks → [embedding model] → vectors (in a store)
                                                              ↓
          query → [embedding model] → vector  → [similarity search]
                                                              ↓
                              top-K chunks + query → [LLM] → grounded answer
```

Every knob matters:

- **Chunk size** — too small = missing context, too big = diluted signal. 256–512 tokens is typical.
- **Embedding model** — BGE, E5, Jina, and OpenAI's ada-002 are common. All produce ~768-1024-dim vectors.
- **Retrieval K** — top-3 to top-10 chunks is standard. More = more context but more noise.
- **Vector store** — in-memory matrix (this lab), FAISS, Milvus, pgvector. Trade-off: latency vs scale.
- **Prompt template** — how you inject the retrieved context affects the LLM's willingness to use it.

### References to read alongside

- **[RAG paper (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)** — the original. Still the cleanest articulation of why retrieval + generation outperforms either alone.
- **[BGE embeddings (Xiao et al., 2023)](https://arxiv.org/abs/2309.07597)** — the embedding model family we'll use. Open, top of MTEB for months.
- **[DPR (Karpukhin et al., 2020)](https://arxiv.org/abs/2004.04906)** — Dense Passage Retrieval; trained embeddings specifically for QA retrieval.
- **[ColBERT (Khattab & Zaharia, 2020)](https://arxiv.org/abs/2004.12832)** — late-interaction retrieval. Better accuracy at the cost of storage.
- **[Lost in the Middle (Liu et al., 2023)](https://arxiv.org/abs/2307.03172)** — classic empirical finding: LLMs use info at the start and end of context, ignore the middle. Implications for RAG prompt design.

### Models

- **Embedding**: `BAAI/bge-small-en-v1.5` (~130 MB, 384-dim vectors, produced by the BGE team)
- **Generator**: `TinyLlama-1.1B-Chat` (~2 GB, used throughout this course)

---

## Step 1 — Build a corpus + load the embedding model

Our knowledge base: 8 carefully-chosen facts about modern LLMs. This is tiny but lets you see RAG end-to-end cleanly. In production your corpus is tens of thousands to millions of documents and the retrieval step has to be fast (milliseconds).

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda')

documents = [
    "Llama 3 was released by Meta in April 2024. It was trained on approximately 15 trillion tokens — roughly 5x more data than Llama 2. The 8B and 70B variants are open-weight.",
    "The Transformer architecture was introduced by Vaswani et al. in the 2017 paper 'Attention Is All You Need'. It replaced recurrence and convolution with pure attention, kicking off the modern LLM era.",
    "GPT-3, released by OpenAI in 2020, has 175 billion parameters. Its emergent few-shot learning abilities surprised researchers and marked a major scaling-law inflection point.",
    "LoRA (Low-Rank Adaptation) was proposed by Hu et al. in 2021. It lets you fine-tune a frozen pretrained model by adding tiny low-rank matrices on top of attention layers, typically cutting trainable parameters by 10000x.",
    "The BGE family of embedding models was released by the Beijing Academy of Artificial Intelligence in 2023. BGE-small has 33 million parameters and produces 384-dimensional vectors.",
    "Mixture of Experts (MoE) models use many specialist subnetworks and a router that sends each token to a subset. Mixtral 8x7B (released December 2023 by Mistral) demonstrated MoE at competitive quality.",
    "The Chinchilla paper (Hoffmann et al., 2022) established compute-optimal training: models should see roughly 20 tokens per parameter during pretraining. Most pre-Chinchilla models were undertrained.",
    "Retrieval-Augmented Generation was formalized in the 2020 RAG paper by Lewis et al. It combines a parametric language model with a non-parametric knowledge store accessed via dense retrieval.",
]
print(f'{len(documents)} documents in corpus')

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


8 documents in corpus


In [2]:
EMBED_ID = 'BAAI/bge-small-en-v1.5'
print(f'Loading embedding model {EMBED_ID} (~130 MB first time)...')
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_ID)
embed_model = AutoModel.from_pretrained(EMBED_ID).to(device)
embed_model.eval()

# The idiomatic BGE encode: tokenize, forward pass, take [CLS] or mean-pool, then L2-normalize.
# BGE's team recommends [CLS] pooling specifically (not mean pool) for this model family.
def embed(texts, max_len=256):
    """Encode a list of strings into L2-normalized 384-dim vectors on CUDA."""
    enc = embed_tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(device)
    with torch.no_grad():
        out = embed_model(**enc)
    # BGE uses [CLS] pooling: the first token's hidden state at the last layer
    pooled = out.last_hidden_state[:, 0]
    # L2-normalize so cosine similarity reduces to dot product
    return F.normalize(pooled, dim=-1)

# Quick smoke test
v = embed(['What is Llama 3?', 'How big is Llama 3?'])
print(f'Encoded shape: {tuple(v.shape)}, norm: {torch.norm(v, dim=-1).cpu().tolist()}')
similar = (v[0] @ v[1]).item()
print(f'Cosine sim between two Llama-3 questions: {similar:.3f} (should be > 0.8 — they are semantically close)')

Loading embedding model BAAI/bge-small-en-v1.5 (~130 MB first time)...


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Encoded shape: (2, 384), norm: [1.0, 0.9999999403953552]
Cosine sim between two Llama-3 questions: 0.868 (should be > 0.8 — they are semantically close)


In [3]:
from preporato_labs import Lab
lab = Lab('rag-pipeline')
lab.check(1)

OK — corpus: 8 documents; embedding model outputs 384-dim L2-normalized vectors
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Chunk the documents, embed the chunks

For a small corpus like ours, each document is already short enough to embed as a whole. Real corpora (PDFs, webpages) need a **chunking strategy**:

- **Fixed-size** — 256-tokens with 32-token overlap. Simplest, works fine for most text.
- **Sentence-based** — split at sentence boundaries. Better for prose.
- **Hierarchical** — section → paragraph → sentence. Needed for long technical docs where structure matters.
- **Semantic** — use an embedding model to find topic boundaries. Clever, rarely worth the complexity.

We'll treat each doc as a single chunk (already short). In a real pipeline you'd split long docs first.

### 🐛 Common mistake: forgetting L2 normalization

Cosine similarity = dot product **only if** both vectors are L2-normalized. If you skip normalization, you're computing raw dot products — which grow with vector magnitude. Your "most similar" retrievals become "longest" retrievals. BGE explicitly normalizes for you in their helper functions; using `AutoModel` directly, you must do it yourself.

In [4]:
# For this tiny corpus: chunk == document. One would add an actual chunker for longer docs.
chunks = list(documents)

# Embed every chunk. These live on GPU and never move during retrieval.
passage_embs = embed(chunks)
print(f'Embedded {passage_embs.shape[0]} chunks into {passage_embs.shape[1]}-dim vectors')
print(f'Shape: {tuple(passage_embs.shape)}, device: {passage_embs.device}')
print(f'GPU memory for passage matrix: {passage_embs.numel() * passage_embs.element_size() / 1024:.1f} KB')

Embedded 8 chunks into 384-dim vectors
Shape: (8, 384), device: cuda:0
GPU memory for passage matrix: 12.0 KB


In [5]:
lab.check(2)

OK — 8 chunks embedded into 384-dim vectors on cuda:0
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — Retrieval: top-K nearest chunks

Embed the query, compute cosine similarity against every stored passage, return the top-K highest. At scale you'd use a vector DB with approximate nearest neighbors (HNSW, IVF, SCANN) — millisecond latency on billions of vectors. For 8 chunks, a single matrix-vector product is instant.

**Vector DB options and when to use them:**

| Tool | Good for | Not for |
|------|----------|---------|
| **Numpy/torch matmul** | < 100K vectors | Latency at scale |
| **FAISS** | Self-hosted, 100K–100M | Distributed, ACID |
| **Milvus / Weaviate** | Large-scale production | Simple prototypes |
| **pgvector** | You already have Postgres | Billion-scale |
| **Pinecone / Qdrant Cloud** | Managed, zero-ops | Offline workflows |


In [6]:
def retrieve(query, k=3):
    """Return (top_chunks, cosine_scores) for the query against our in-memory corpus."""
    q_vec = embed([query])                               # (1, 384)
    scores = (q_vec @ passage_embs.T).squeeze(0)         # (n_chunks,)
    top_scores, top_idx = torch.topk(scores, k=min(k, len(chunks)))
    return [chunks[i] for i in top_idx.tolist()], top_scores.cpu().tolist()

query = 'How many tokens was Llama 3 trained on?'
top_chunks, similarity_scores = retrieve(query, k=3)

retrieval_test = {'query': query, 'top_chunks': top_chunks, 'similarity_scores': similarity_scores}

print(f'QUERY: {query!r}\n')
for i, (chunk, score) in enumerate(zip(top_chunks, similarity_scores), 1):
    print(f'[{i}] score={score:.3f}')
    print(f'    {chunk[:200]}...' if len(chunk) > 200 else f'    {chunk}')
    print()

QUERY: 'How many tokens was Llama 3 trained on?'

[1] score=0.829
    Llama 3 was released by Meta in April 2024. It was trained on approximately 15 trillion tokens — roughly 5x more data than Llama 2. The 8B and 70B variants are open-weight.

[2] score=0.696
    The Chinchilla paper (Hoffmann et al., 2022) established compute-optimal training: models should see roughly 20 tokens per parameter during pretraining. Most pre-Chinchilla models were undertrained.

[3] score=0.615
    GPT-3, released by OpenAI in 2020, has 175 billion parameters. Its emergent few-shot learning abilities surprised researchers and marked a major scaling-law inflection point.



In [7]:
lab.check(3)

OK — query 'How many tokens was Llama 3 trained on?' retrieved top-3 chunks; best similarity 0.829
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Generate with and without retrieval context

The moment of truth: ask the LLM the same question with and without retrieved context, compare.

The prompt template matters — we'll use a classic RAG pattern:

```
Use the following context to answer the question.

CONTEXT: <retrieved chunks>

QUESTION: <user query>

ANSWER:
```

### The 'Lost in the Middle' effect

[Liu et al. 2023](https://arxiv.org/abs/2307.03172) showed that when you put many passages into context, the LLM reliably attends to the first and last but tends to **skip the middle**. Two mitigations used in production:

- Re-rank retrieved chunks so the most relevant end up at the top
- Include fewer, higher-quality passages rather than many mediocre ones

For 3 chunks this barely matters. For top-20 chunks it becomes critical.

In [8]:
print('Loading TinyLlama-1.1B-Chat for generation...')
GEN_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_ID)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_ID, torch_dtype=torch.float16).to(device)
gen_model.eval()

def generate_answer(prompt, max_new=120):
    ids = gen_tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = gen_model.generate(
            ids, max_new_tokens=max_new, do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    return gen_tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

print('OK — generator loaded')

Loading TinyLlama-1.1B-Chat for generation...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

OK — generator loaded


In [9]:
# No-RAG: just ask the question
no_rag_prompt = f"Q: {query}\nA:"
no_rag_answer = generate_answer(no_rag_prompt)

# RAG: inject retrieved context
context = '\n\n'.join(top_chunks)
rag_prompt = (
    f"Use the following context to answer the question.\n\n"
    f"CONTEXT:\n{context}\n\n"
    f"QUESTION: {query}\n\n"
    f"ANSWER:"
)
rag_answer = generate_answer(rag_prompt)

print('=' * 70)
print(f'QUERY: {query}')
print('=' * 70)
print(f'\nNO-RAG ANSWER (TinyLlama alone):')
print(f'  {no_rag_answer}')
print(f'\nRAG ANSWER (same LLM + retrieved context):')
print(f'  {rag_answer}')
print('\n' + '=' * 70)
print('The RAG answer should reference 15 trillion tokens (from our corpus). The no-RAG answer')
print('may hallucinate or give a generic response — TinyLlama-1.1B has no memory of Llama 3 numbers.')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


QUERY: How many tokens was Llama 3 trained on?

NO-RAG ANSWER (TinyLlama alone):
  Llama 3 was trained on 3000 tokens.

RAG ANSWER (same LLM + retrieved context):
  Llama 3 was trained on approximately 15 trillion tokens.

The RAG answer should reference 15 trillion tokens (from our corpus). The no-RAG answer
may hallucinate or give a generic response — TinyLlama-1.1B has no memory of Llama 3 numbers.


In [10]:
lab.check(4)

OK — RAG answer differs from no-RAG baseline:
   NO-RAG: 'Llama 3 was trained on 3000 tokens.'
   RAG:    'Llama 3 was trained on approximately 15 trillion tokens.'
STEP_PASSED


Step 4 Complete! Lab complete!

True

---

## What you just built

The complete RAG pipeline: corpus → chunker → embedding → vector store → retrieval → grounded generation. Swap any piece (bigger corpus, better embedding model, production vector DB, bigger LLM) and the pipeline scales unchanged to enterprise-grade.

## What to read next

- **[RAG paper (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)** — the origin.
- **[BGE paper (Xiao et al., 2023)](https://arxiv.org/abs/2309.07597)** — top open embedding models for months running.
- **[Lost in the Middle (Liu et al., 2023)](https://arxiv.org/abs/2307.03172)** — where long context goes to die.
- **[Self-RAG (Asai et al., 2023)](https://arxiv.org/abs/2310.11511)** — teach the LLM to decide when to retrieve.
- **[LangChain RAG docs](https://python.langchain.com/docs/tutorials/rag/)** and **[LlamaIndex RAG docs](https://docs.llamaindex.ai/)** — the two dominant RAG orchestration frameworks.
- **[MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard)** — pick your embedding model here. NV-Embed-v2, BGE-M3, Stella-en-v5 all top-tier as of 2025.

## What to try next

- Replace the in-memory matrix with **FAISS** (`faiss-gpu`) or **Milvus Lite** (`pymilvus`) and measure retrieval latency on 10K+ chunks.
- Try a **cross-encoder re-ranker** (e.g. `BAAI/bge-reranker-base`) to re-order retrieved chunks before sending to the LLM.
- Swap TinyLlama for **Qwen2.5-7B-Instruct** in NF4 (Lab 3) — the RAG answer quality will improve dramatically.
- Build **hybrid search**: combine BM25 (keyword) with dense retrieval and take a weighted blend. Canonical in production.